# Буткемп, день 12 — cross-venue market making: quote here, hedge there

Три роли, которые в этой задаче разведены по трём разным биржам: **quote venue** $M$ — биржа, на которой мы
выставляем котировки и снимаем спред; **leader venue** $L$ — биржа, чей microprice мы берём якорем справедливой
цены; **hedge venue** $H$ — биржа, на которой мы снимаем ликвидность, чтобы выровнять позицию, накопленную
филлами на $M$. Формула дня: лидер говорит цену, фолловер позволяет по ней котироваться, а хедж-биржа
даёт возможность удержать спред. У каждой из трёх ролей есть своя цена — у лида затухание сигнала со временем, у якоря
latency, у хеджа стоимость прохода по стакану, — и всё, что мы сегодня считаем про деньги, условно при этих ценах.

Задача — построить кросс-венью движок котирования на L2-данных трёх бирж, просимулировать его с явной
моделью филлов и **докладывать каждый результат как условный при допущенных параметрах филлов**. Ручки такие:

1. **Measure** — по часам: кто лидер и как затухает лид; лид microprice над mid других бирж; basis и его скачки;
   hedge cost curve каждой биржи.
2. **Quote engine** — quote-функция и hedge-правило **чистыми функциями** с unit-тестами; оценка basis с
   заявленным half-life и break detector.
3. **Simulate** — на реальных L2-путях, филлы явной моделью со sweep $A$ и $k$; якоря own mid / leader mid /
   leader microprice; data latency $0$–$500$ мс.
4. **Hedge** — политики every-fill и threshold; certainty equivalent при $\gamma=0.01,\,0.03,\,0.10$; sweep
   hedge delay.
5. **Break it** — basis jump, stale leader, thin hedge book; показать работу stand-down-протокола.
6. **Report** — что измерено и что допущено, чувствительность, пройденные тесты, почему результат —
   верхняя граница.

Данные — те же, что в предыдущие дни. Первый файл — `l2_executable_quotes_100ms_10000.parquet`: top-of-book на
сетке 100 мс за 48 часов по трём биржам (Bybit, Okex, Binance) и двум активам (BTC, ETH), с объёмами на лучших
ценах. Именно эти объёмы (`best_bid_amount`, `best_ask_amount`) позволяют считать microprice — без них якорь
из microprice не построить. Для **hedge cost curve** top-of-book недостаточно: нужно пройти по нескольким
уровням стакана, поэтому берём три полных файла `bybit_l2_/okex_l2_/binance_l2_20260701_20260703.parquet` со
схемой `symbol, event_time, side, price, amount`. Тейпа (записи сделок) и собственных филлов у нас нет, поэтому
интенсивности $A$ и $k$ — это **допущения**, а не оценки; всё P&L ниже — условное. Работаем по BTC.

Сразу оговорю то, что дальше видно по числам: реальный спред BTC равен одному тику ($0.1$, то есть $\approx
0.017$ bp), а все эффекты дня — десятые доли bp. Масштаб микроструктурный; числа надо трактовать как условные при выбранных $(A,k)$, а не как обещание прибыли.

Отдельно и сразу: **лабораторные числа слайда** ($1773$/$2564$/$2487$ за час по якорям, $-55\%$ за рассогласование
горизонта) — иллюстративный пример с другими $(A,k,\beta,\sigma)$; на наших реальных траекториях они не воспроизводятся,
и мы этого совпадения не ищем. И про лидерство, тоже «сразу»: на сетке 100 мс **отличимого лидера между биржами нет** — лаговый сигнал
оказывается артефактом сетки (разведка покажет три проверки), поэтому «лидерский» якорь в симуляции —
иллюстративный выбор fair, а не выведенное из данных свойство Bybit. Дальше сходимся на качественной
структуре эффектов (якорь $=$ fair $\gg$ свой mid, microprice —
второй порядок, threshold $\ge$ every-fill), а абсолютные величины приводим своими прогонами.

## Разведка данных

Начнём с того, что лежит в каждом файле, и с грабли, которая в этой задаче стоит дороже всего: **конвенция
`side`**. В полных стаканах `side=1` — это bid (заявки покупателей), `side=-1` — ask (заявки продавцов).
Перепутать знак значит перевернуть весь walk по стакану и получить стоимость хеджа с точностью до знака,
поэтому проверяем соглашение явно: лучшая цена на стороне `1` должна быть *ниже* лучшей цены на стороне `-1`.

In [1]:
import numpy as np, pandas as pd, duckdb, warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

Q  = "/Users/b.kotov/Downloads/l2_executable_quotes_100ms_10000.parquet"
FULL = {
    "Bybit":   ("/Users/b.kotov/Downloads/bybit_l2_20260701_20260703.parquet",   "BTCUSDT"),
    "Okex":    ("/Users/b.kotov/Downloads/okex_l2_20260701_20260703.parquet",     "BTC-USDT-SWAP"),
    "Binance": ("/Users/b.kotov/Downloads/binance_l2_20260701_20260703.parquet", "BTCUSDT"),
}
con = duckdb.connect()

head = con.execute(f"SELECT exchange, asset, event_time, best_bid, best_ask, best_bid_amount, best_ask_amount, mid, spread_bps FROM read_parquet('{Q}') LIMIT 3").df()
info = con.execute(f"SELECT count(*) n, min(event_time) t0, max(event_time) t1, count(DISTINCT exchange) nex, count(DISTINCT asset) nas FROM read_parquet('{Q}')").df()
print("top-of-book (3 строки):")
print(head.to_string(index=False))
print("\nвсего строк:", int(info.n[0]), "| интервал:", info.t0[0], "->", info.t1[0], "| бирж:", int(info.nex[0]), "| активов:", int(info.nas[0]))

print("\nполные стаканы (символ, строк):")
for ex in FULL:
    p, sym = FULL[ex]
    r = con.execute(f"SELECT count(*) n FROM read_parquet('{p}') WHERE symbol='{sym}'").df()
    print(" ", ex.ljust(8), sym.ljust(14), f"{int(r.n[0]):>12,} строк")

top-of-book (3 строки):
exchange asset                       event_time  best_bid  best_ask  best_bid_amount  best_ask_amount       mid  spread_bps
   Bybit   BTC 2026-07-01 03:00:00.083350+03:00  58603.40  58603.50            3.977            3.207 58603.450    0.017064
    Okex   ETH 2026-07-01 03:00:00.089859+03:00   1571.35   1571.40          139.728            0.580  1571.375    0.318193
   Bybit   ETH 2026-07-01 03:00:00.093596+03:00   1571.20   1571.21           39.980            0.070  1571.205    0.063645

всего строк: 10244979 | интервал: 2026-07-01 03:00:00.083350+03:00 -> 2026-07-03 02:59:59.896158+03:00 | бирж: 3 | активов: 2

полные стаканы (символ, строк):
  Bybit    BTCUSDT          63,175,047 строк


  Okex     BTC-USDT-SWAP   209,219,709 строк


  Binance  BTCUSDT         369,331,742 строк


Проверим конвенцию `side` на снимке стакана. Снимок в момент $T$ — это «последнее известное состояние»
каждого уровня цены: берём по каждой паре `(side, price)` событие с максимальным `event_time` до $T$ и
оставляем те, где итоговый `amount > 0` (нулевой `amount` означает снятие уровня, а не нулевую заявку). Bybit
на самом старте периода отдаёт лишь несколько уровней, поэтому снимок берём через минуты после начала, когда
стакан набран полностью.

In [2]:
def snapshot(ex, minutes=5):
    path, sym = FULL[ex]
    return con.execute(f"""
        WITH win AS (SELECT side, price, amount, event_time FROM read_parquet('{path}')
                     WHERE symbol='{sym}'
                       AND event_time < (SELECT min(event_time) + INTERVAL {minutes} MINUTE
                                         FROM read_parquet('{path}') WHERE symbol='{sym}'))
        SELECT side, price, arg_max(amount, event_time) AS amount
        FROM win GROUP BY side, price
        HAVING arg_max(amount, event_time) > 0
    """).df()

print("конвенция side: best price на стороне 1 (bid) должна быть ниже стороны -1 (ask)")
for ex in ["Bybit","Okex","Binance"]:
    bk = snapshot(ex)
    bb = bk[bk.side==1].price.max(); ba = bk[bk.side==-1].price.min()
    ok = "ok" if bb < ba else "ПЕРЕВЁРНУТО / битый снимок"
    print("  " + ex.ljust(8) + f"levels={len(bk):5d}  best_bid(side=1)={bb:>10}  best_ask(side=-1)={ba:>10}  -> {ok}")

конвенция side: best price на стороне 1 (bid) должна быть ниже стороны -1 (ask)


  Bybit   levels=  100  best_bid(side=1)=   58706.3  best_ask(side=-1)=   58706.4  -> ok


  Okex    levels=  800  best_bid(side=1)=   58698.9  best_ask(side=-1)=   58699.0  -> ok


  Binance levels=10837  best_bid(side=1)=   58699.9  best_ask(side=-1)=   58612.8  -> ПЕРЕВЁРНУТО / битый снимок


Bybit и Okex соглашение подтверждают: сторона `1` собрана ниже стороны `-1`. **Binance полного стакана
не отдаёт корректно** — в её файле смешаны уровни с дикими ценами (bid на $1000$ и ask на $150000$), то есть
снимок «последнее событие на уровне» содержит уровни давно умерших инструментов или битые записи. Поэтому полный стакан Binance для walk по книге непригоден, и hedge cost curve мы строим по Bybit и Okex —
там книга чистая. Это дефект именно файла полного стакана (история уровней за 48 часов с мусором),
а не биржи — top-of-book Binance из `l2_executable_quotes` исправен, и lead-lag, basis, симуляции на Binance
мы по нему считаем как обычно.

Теперь основная разведка пункта 1. Сводим три биржи на общую сетку 100 мс (для каждого такта берём последний
mid, сетку регуляризуем `floor("100ms")` и `last()`), считаем лог-доходности и смотрим, кто кого ведёт.

## Гипотеза дня и её проверка

Гипотеза ровно та, ради которой затевается кросс-венью: **наш собственный mid «протухает» — фолловер
отстаёт, поэтому якорем справедливой цены должен быть лидер, а не наша биржа**. Если это так, котирование
вокруг собственного mid систематически ставит цены «не там», а вокруг лидера — «там», и
разница в P&L на филл даётся отклонением якоря от истинной цены $e = \text{mid}_L - \text{anchor}$, которое
входит в ожидаемый P&L двумя членами $\pm e\cdot k$.

Проверяем числом: находим лидера по lead-lag (кто ведёт остальных на горизонте 100 мс сильнее), а потом
прогоняем симулятор с якорем на своём mid, на mid лидера и на microprice лидера. Если лидер различим, ждём,
что P&L на лидерском якоре уверенно обгоняет якорь на своём mid, и что разрыв растёт с latency. Заодно
проверка обязана ответить и на базовый вопрос — различим ли лидер вообще на сетке 100 мс; если нет,
«лидерский» якорь в симуляции станет иллюстративным выбором fair.

## Пункт 1. Measure — лидер, затухание, basis, hedge cost curve

Lead-lag меряем как корреляцию лог-доходностей: $\rho_{a\to b}(\ell) = \mathrm{corr}\big(\Delta\ln m^a_t,\;
\Delta\ln m^b_{t+\ell}\big)$ — метка $a\to b$ читается «$a$ ведёт $b$»: движение $a$ сейчас коррелирует
с будущим движением $b$. Рядом печатаем контемпоранную корреляцию $\ell=0$ — без неё лаговую цифру читать
нельзя. Считаем на всей выборке и отдельно по часам, чтобы увидеть, устойчив ли лидер.

In [3]:
TICK = 0.1
df = con.execute(f"""SELECT exchange, asset, event_time, mid,
        best_bid, best_ask, best_bid_amount, best_ask_amount
    FROM read_parquet('{Q}') WHERE asset='BTC' AND mid IS NOT NULL ORDER BY event_time""").df()
df["t"] = pd.to_datetime(df.event_time, utc=True).dt.floor("100ms")
df["micro"] = (df.best_ask*df.best_bid_amount + df.best_bid*df.best_ask_amount) / (df.best_bid_amount + df.best_ask_amount)
g = df.groupby(["exchange","t"]).agg(mid=("mid","last"), micro=("micro","last")).reset_index()
mid = g.pivot(index="t", columns="exchange", values="mid").dropna()
miP = g.pivot(index="t", columns="exchange", values="micro").dropna()
print("общих тактов на 100ms сетке:", len(mid), "| колонки:", list(mid.columns))

lr = np.log(mid).diff()
PAIRS = [("Binance","Bybit"), ("Okex","Bybit"), ("Binance","Okex")]
print("\nlead-lag по всему периоду, corr(dlog mid). Метка X->Y при лаге L>0 читается «X ведёт Y»: corr(X_t, Y_{t+L}).")
print("   L=0 — контемпоранная корреляция corr(X_t, Y_t), референс:")
print("   ", {f"{x}~{y}": round(lr[x].corr(lr[y]), 3) for x, y in PAIRS})
for L in [1,2,3,5]:
    row = {f"{x}->{y}": round(lr[x].corr(lr[y].shift(-L)), 3) for x, y in PAIRS}
    row.update({f"{y}->{x}": round(lr[y].corr(lr[x].shift(-L)), 3) for x, y in PAIRS})
    print("  ", L*100, "ms:", row)


общих тактов на 100ms сетке: 1666616 | колонки: ['Binance', 'Bybit', 'Okex']

lead-lag по всему периоду, corr(dlog mid). Метка X->Y при лаге L>0 читается «X ведёт Y»: corr(X_t, Y_{t+L}).
   L=0 — контемпоранная корреляция corr(X_t, Y_t), референс:
    {'Binance~Bybit': np.float64(0.407), 'Okex~Bybit': np.float64(0.602), 'Binance~Okex': np.float64(0.538)}


   100 ms: {'Binance->Bybit': np.float64(0.595), 'Okex->Bybit': np.float64(0.341), 'Binance->Okex': np.float64(0.361), 'Bybit->Binance': np.float64(0.1), 'Bybit->Okex': np.float64(0.119), 'Okex->Binance': np.float64(0.126)}
   200 ms: {'Binance->Bybit': np.float64(0.101), 'Okex->Bybit': np.float64(0.07), 'Binance->Okex': np.float64(0.074), 'Bybit->Binance': np.float64(0.042), 'Bybit->Okex': np.float64(0.049), 'Okex->Binance': np.float64(0.052)}


   300 ms: {'Binance->Bybit': np.float64(0.035), 'Okex->Bybit': np.float64(0.031), 'Binance->Okex': np.float64(0.031), 'Bybit->Binance': np.float64(0.032), 'Bybit->Okex': np.float64(0.028), 'Okex->Binance': np.float64(0.032)}
   500 ms: {'Binance->Bybit': np.float64(0.017), 'Okex->Bybit': np.float64(0.015), 'Binance->Okex': np.float64(0.02), 'Bybit->Binance': np.float64(0.015), 'Bybit->Okex': np.float64(0.018), 'Okex->Binance': np.float64(0.017)}


При корректной метке направление читается так: $\mathrm{corr}(\Delta\ln m^{Binance}_t,\Delta\ln m^{Bybit}_{t+1})=0.595$ — формально «Binance ведёт Bybit» на одном такте, обратное направление лишь $0.10$; к 200 мс всё падает до $\approx0.1$. Для микроструктуры $0.6$ — величина крупная ($R^2\approx36\%$), но уже здесь виден тревожный профиль: лаговая корреляция *выше* контемпоранной ($0.407$), чего у честного лида быть не должно — у него максимум при $L=0$ и убывание с лагом. Что это значит, разберём на диагностике после почасовой проверки; сначала устойчивость по часам.


In [4]:
print("lead-lag по часам (лаг 100 ms):")
rows = []
for h, gg in lr.groupby(lr.index.floor("h")):
    c = {}
    for a in mid.columns:
        for b in mid.columns:
            if a < b:
                c[a + "->" + b] = gg[a].corr(gg[b].shift(-1))
                c[b + "->" + a] = gg[b].corr(gg[a].shift(-1))
    rows.append({"hour": str(h)[11:16], **{k2: round(v,3) for k2,v in c.items()}})
hl = pd.DataFrame(rows)
print("  часов:", len(hl))
print(hl.describe().loc[["mean","std","min","max"]].round(3).to_string())

lead-lag по часам (лаг 100 ms):


  часов: 48
      Binance->Bybit  Bybit->Binance  Binance->Okex  Okex->Binance  Bybit->Okex  Okex->Bybit
mean           0.580           0.097          0.344          0.123        0.117        0.335
std            0.065           0.025          0.053          0.030        0.028        0.046
min            0.308           0.042          0.245          0.062        0.059        0.152
max            0.703           0.161          0.470          0.199        0.181        0.436


In [5]:
print("=== диагностика: разрешается ли лид на сетке 100 мс ===")
print("1) Профиль по лагам. У честного лида максимум при L=0 и убывание с лагом. Binance~Bybit:")
c0 = lr["Binance"].corr(lr["Bybit"]); c1 = lr["Binance"].corr(lr["Bybit"].shift(-1))
print("   L=0: %.3f | L=1: %.3f -> лаговая ВЫШЕ контемпоранной: это артефакт сетки, не лид" % (c0, c1))

print("2) На 1s-сетке лаговый сигнал исчезает, контемпоранная остаётся:")
df["t1"] = pd.to_datetime(df.event_time, utc=True).dt.floor("1s")
mid1 = df.groupby(["exchange","t1"]).agg(mid=("mid","last")).reset_index() \
        .pivot(index="t1", columns="exchange", values="mid").dropna()
lr1 = np.log(mid1).diff()
for x, y in PAIRS:
    print("   %s~%s: L=0 %.3f | L=1 %s->%s %.3f | %s->%s %.3f" % (x, y, lr1[x].corr(lr1[y]),
          x, y, lr1[x].corr(lr1[y].shift(-1)), y, x, lr1[y].corr(lr1[x].shift(-1))))

print("3) Знак 100-мс приращений совпадает в ~90% тактов (биржи ко-движутся):")
for x, y in PAIRS:
    m = lr[x].notna() & lr[y].notna()
    agree = (np.sign(lr[x][m]) == np.sign(lr[y][m])).mean()
    nz = m & (lr[x] != 0) & (lr[y] != 0)
    print("   %s vs %s: %.1f%% всех | %.1f%% ненулевых" % (x, y, agree*100,
          (np.sign(lr[x][nz]) == np.sign(lr[y][nz])).mean()*100))

us = pd.to_datetime(df.event_time, utc=True).dt.microsecond
mu = df.assign(us=us).groupby("exchange")["us"].mean()
print("4) Средние мкс-доли event_time почти равны (заметного смещения часов нет):",
      {k: int(v) for k, v in mu.items()})


=== диагностика: разрешается ли лид на сетке 100 мс ===
1) Профиль по лагам. У честного лида максимум при L=0 и убывание с лагом. Binance~Bybit:
   L=0: 0.407 | L=1: 0.595 -> лаговая ВЫШЕ контемпоранной: это артефакт сетки, не лид
2) На 1s-сетке лаговый сигнал исчезает, контемпоранная остаётся:


   Binance~Bybit: L=0 0.897 | L=1 Binance->Bybit 0.110 | Bybit->Binance 0.083
   Okex~Bybit: L=0 0.888 | L=1 Okex->Bybit 0.097 | Bybit->Okex 0.089
   Binance~Okex: L=0 0.886 | L=1 Binance->Okex 0.110 | Okex->Binance 0.091
3) Знак 100-мс приращений совпадает в ~90% тактов (биржи ко-движутся):
   Binance vs Bybit: 90.0% всех | 93.6% ненулевых
   Okex vs Bybit: 91.3% всех | 96.6% ненулевых
   Binance vs Okex: 91.6% всех | 96.2% ненулевых


4) Средние мкс-доли event_time почти равны (заметного смещения часов нет): {'Binance': 540663, 'Bybit': 529190, 'Okex': 539800}


Картина по часам устойчива: Binance->Bybit в среднем $0.58$ при разбросе по часам $\approx0.07$; в каждой
паре находится «сильное» направление (Binance->Okex и Okex->Bybit по $\approx0.34$ в среднем по часам), а все
обратные держатся на $\approx0.10$–$0.12$. Минимум по часам заметно выше нуля — структура лагов жива все 48 часов. При
корректной метке «впереди» оказывается Binance, а не Bybit, — уже это подсказывает, что на такую цифру как на
экономический лид опираться нельзя. Диагностика выше это подтверждает, и вывод по вопросу «who leads» такой:
**отличимого лидера на этих данных нет**. Три аргумента. Первый: на 100 мс лаговая корреляция ($0.595$)
*выше* контемпоранной ($0.407$) — у честного лида максимум стоит при $L=0$ и убывает с лагом, перевёрнутый
профиль бывает только у артефакта сетки и шума. Второй: на 1s-сетке лаговый сигнал исчезает ($0.10$–$0.11$
против $0.89$–$0.90$ контемпоранной) — экономический лид при укрупнении такта не испаряется. Третий: знак
100-мс приращений совпадает в 90% тактов (на ненулевых — 94–97%), а микросекундные доли `event_time` у бирж
почти равны — биржи двигаются *одновременно*, без заметного смещения часов. Значит, домен настоящего лида
меньше 100 мс — за разрешением имеющихся файлов, и ответ на слайдный вопрос тот же: **отличимого лидера нет**.
Для симуляции это меняет статус выводов: «лидерский» якорь ниже — это mid конкретной биржи (Bybit), выбранная
точка отсчёта, а не выведенное из данных свойство. Отдельный сигнал — microprice: он предсказывает
будущий mid *внутри* биржи и к межбирженному лидерству не относится.

Теперь microprice. Это цена, взвешенная дисбалансом объёмов на лучших ценах:

$$\text{micro} = \frac{a\,V_b + b\,V_a}{V_b + V_a} = \text{mid} + \frac{s}{2}\,(2I-1),\qquad
I = \frac{V_b}{V_b+V_a},$$

где $b,a$ — лучшие bid/ask, $V_b,V_a$ — их объёмы, $s=a-b$ — спред. Тождество справа — проверка реализации:
счёт через imbalance $I$ должен сойтись с прямой формулой до машинной точности. И главное — microprice должен
**предсказывать** будущий mid: если объём перевешивает на bid, цена скорее пойдёт вверх.

In [6]:
sp = (df.best_ask - df.best_bid)/df.mid*1e4
I  = df.best_bid_amount/(df.best_bid_amount + df.best_ask_amount)
ident = df.mid*(1 + (sp/1e4)/2*(2*I-1))
print("тождество micro = mid + spread/2*(2I-1):  max abs err =", f"{np.max(np.abs(df.micro - ident)/df.mid*1e4):.2e} bp")

print("\nлид microprice над mid: corr( (micro-mid)/mid , dlog mid на h вперёд )")
res = []
for ex in ["Bybit","Okex","Binance"]:
    d = g[g.exchange==ex].set_index("t")
    dev = (d.micro - d.mid)/d.mid
    row = {"exchange": ex}
    for h in [1,2,5,10]:
        fwd = np.log(d.mid.shift(-h)/d.mid)
        row[str(h*100)+"ms"] = dev.corr(fwd)
    res.append(row)
print(pd.DataFrame(res).round(3).to_string(index=False))

тождество micro = mid + spread/2*(2I-1):  max abs err = 4.99e-12 bp

лид microprice над mid: corr( (micro-mid)/mid , dlog mid на h вперёд )


exchange  100ms  200ms  500ms  1000ms
   Bybit  0.175  0.158  0.139   0.124
    Okex  0.141  0.117  0.093   0.083
 Binance  0.092  0.092  0.079   0.073


Тождество сходится до $5\cdot 10^{-12}$ bp — реализация microprice верна. Лид microprice над mid
положителен на всех биржах и убывает с горизонтом: Bybit $+0.175$ на 100 мс, $+0.124$ на 1 с; у Binance
вдвое слабее ($+0.092$). Знак положительный — microprice указывает сторону будущего движения, и сильнее всего
у Bybit. Это предсказуемость mid *внутри* биржи, а не лидерство между биржами (см. диагностику выше). $0.175$ — заметная для микроструктуры сила ($R^2\approx3\%$), но небольшая и быстро
рассеивающаяся: к 2–5 секундам остаётся около $0.1$. Это ограничивает горизонт, на котором якорь из microprice
имеет смысл, — отсюда связь с hedge delay.

Дальше basis. Basis — это **не сигнал**, а стойкий сдвиг между биржами: разница mid, нормированная на цену.
Меряем относительно Bybit — как референса, а не как выведенного лидера (см. оговорку выше) — и смотрим, насколько он стабилен и как скачет.

In [7]:
bp = 1e4
basis = pd.DataFrame({
    "Okex":    (mid["Okex"]    - mid["Bybit"])/mid["Bybit"]*bp,
    "Binance": (mid["Binance"] - mid["Bybit"])/mid["Bybit"]*bp,
})
st = basis.agg(["mean","std"]).T
st["p01"] = basis.quantile(0.01); st["p99"] = basis.quantile(0.99)
print("basis к Bybit (bp):")
print(st.round(4).to_string())

d1 = np.abs(basis.diff())
print("\nскачки |d basis| за 100 ms (bp):")
jump = pd.DataFrame({"p99": d1.quantile(0.99), "p99.9": d1.quantile(0.999), "max": d1.max(),
                     "share>0.5bp": (d1>0.5).mean(), "share>1bp": (d1>1.0).mean()})
print(jump.round(4).to_string())

basis к Bybit (bp):
           mean     std     p01     p99
Okex    -0.1766  0.8263 -1.9610  1.9508
Binance -0.1917  0.8447 -1.8413  2.1984

скачки |d basis| за 100 ms (bp):


            p99   p99.9      max  share>0.5bp  share>1bp
Okex     0.9475  1.7462  13.3134       0.0405     0.0083
Binance  1.0707  2.2039  19.0862       0.0493     0.0122


Basis устойчивый и отрицательный: Okex в среднем на $-0.18$ bp, Binance на $-0.19$ bp ниже Bybit, при
$sd\approx 0.83$ bp и хвостах $p_1/p_{99}$ около $\mp 2$ bp. Отрицательный mean означает, что Bybit
торгуется чуть дороже — это структурный, не случайный сдвиг. Но `sd` вчетверо больше самого сдвига: большую
часть времени basis «дышит» вокруг уровня, поэтому его надо оценивать, а не брать константой. Отдельно скачки: $p_{99}$
абсолютного приращения за 100 мс равен $\approx 1.07$ bp, а бывают выбросы и на $19$ bp — basis может
перескочить почти мгновенно, и это отдельная ловушка для оценщика.

Теперь hedge cost curve — сколько стоит **снять** ликвидность. Идём по стакану от лучшей цены и набираем
$u$ единиц; стоимость — средневзвешенная по объёму цена прохода. Отклонение средней цены исполнения от mid,
нормированное в bp, и есть цена хеджа на единицу:

In [8]:
def walk(book, side_val, units, mid):
    "средняя цена прохода `units` единиц по стороне; off от mid в bp (>0 — платим спред)"
    b = book[book.side==side_val].sort_values("price", ascending=(side_val < 0))
    rem, cost = units, 0.0
    for p, a in zip(b.price.to_numpy(), b.amount.to_numpy()):
        take = min(a, rem); cost += take*p; rem -= take
        if rem <= 1e-9: break
    if rem > 1e-9: return np.nan           # книга тоньше запрошенного размера
    avg = cost/units
    return (avg-mid)/mid*bp if side_val < 0 else (mid-avg)/mid*bp

BOOKS, MIDS = {}, {}
for ex in ["Bybit","Okex"]:
    bk = snapshot(ex); BOOKS[ex] = bk
    MIDS[ex] = (bk[bk.side==1].price.max() + bk[bk.side==-1].price.min())/2

UNITS = [1,2,4,8,12,20]
print("hedge cost curve (проход по ask), off от mid в bp:")
for ex in ["Bybit","Okex"]:
    bk, m = BOOKS[ex], MIDS[ex]
    vals = [walk(bk, -1, u, m) for u in UNITS]
    print(" ", ex.ljust(8) + ":", "  ".join(f"{u}:{v:+.4f}" for u, v in zip(UNITS, vals)))
    lim = m*(1+0.0001)
    print("            глубина ask до 0.01% от mid:", round(bk[(bk.side==-1)&(bk.price<=lim)].amount.sum(),1), "units")

hedge cost curve (проход по ask), off от mid в bp:
  Bybit   : 1:+0.0085  2:+0.0085  4:+0.0085  8:+0.0085  12:+0.0286  20:+0.1122
            глубина ask до 0.01% от mid: 25.5 units
  Okex    : 1:+0.0085  2:+0.0085  4:+0.0085  8:+0.0740  12:+0.1061  20:+0.1352
            глубина ask до 0.01% от mid: 26.8 units


Кривая почти ступенчата: малые размеры ($1$–$8$ units) идут по **одному тику** — полспреда,
$+0.0085$ bp над mid (mid лежит ровно между bid и ask, поэтому тик $0.017$ bp выглядит как $0.0085$ bp
одной стороны). Только с $\approx 12$ units цена начинает взбираться по книге: $+0.03$, затем $+0.11$ bp
у 20 units. Заявленный размер $1$–$8$ BTC хеджируется практически бесплатно — платим ровно полспреда.
Глубина ask до $0.01\%$ от mid у Bybit $\approx 25$ units (units здесь — BTC), у Okex $\approx 27$ units: запас есть,
но конечен — хедж 20+ units уже толкает цену, а больше $\approx 40$ units потребовал бы идти существенно глубже.
Оговорка к хвосту кривой: снимок берётся в одну минуту, а стакан «дышит», поэтому глубина и стоимость 20+ units
меняются от снимка к снимку на десятые доли bp (в разведке на другом моменте — 27.4 units и $0.082$ bp против
наших 25.5 units и $0.112$ bp); платформа «один тик до $\approx 8$ units» при этом устойчива. Это и есть «цена hedge venue» из эпиграфа.

### Итог разведки

Главная находка — **отличимого лидера на этих данных нет**: формально «впереди» Binance (лаговая корреляция
$0.595$), но она *выше* контемпоранной ($0.407$), на 1s лаговый сигнал исчезает ($0.10$ против $0.89$–$0.90$),
а знак 100-мс приращений совпадает в 90% тактов — биржи движутся синхронно, и домен настоящего лида меньше 100 мс,
за разрешением файлов. Microprice Bybit предсказывает будущий mid с $+0.17$ на 100 мс, но затухает за 1–2 такта —
это про предсказуемость цены внутри биржи, не про лидерство. Basis — сдвиг $\approx0.18$–$0.19$ bp при
$sd\approx0.83$ bp с редкими скачками до $19$ bp. Стоимость хеджа малых размеров — полспреда, крупных — круто
растёт. Эти цены и определяют модель дня; выбор Bybit якорем в симуляции ниже — иллюстративный, а не выведенный
из корреляций.

## Пункт 2. Quote engine — чистые функции, тесты, оценщик basis

Движок раскладывается на две чистые функции без состояния: `quotes(...)` превращает «справедливую» цену,
возраст сигнала и позицию в пару котировок, а `hedge_qty(...)` — позицию и порог в объём хеджа. Полуспред
собирается из четырёх добавок, у каждой свой источник:

$$\text{half} = \underbrace{\tfrac1k}_{\text{competition}} + \underbrace{h}_{\text{hedge cost}}
+ \underbrace{a}_{\text{adverse}} + \underbrace{\lambda\,\text{age}}_{\text{staleness}}, \qquad
\text{skew} = -c\,q\;(q>0 \Rightarrow \text{опускаем}),\qquad \text{bid} = \text{anchor} + \text{skew} - \text{half},\;
\text{ask} = \text{anchor} + \text{skew} + \text{half}.$$

Член $1/k$ — равновесный спред в risk-neutral мире (в 11-м дне видели: при $\lambda(\delta)=Ae^{-k\delta}$ он
не зависит от $A$). $h$ — стоимость хеджа с кривой выше. $a$ — adverse selection (в 11-м дне сдвигало
$\delta^*$ в $1/k+a$). $\lambda\,\text{age}$ — расширение спреда по мере устаревания сигнала: чем дольше
молчит якорный фид, тем мы «слепее» и тем шире должны стоять. `None` возвращаем, когда возраст сигнала превышает
максимально допустимый: это и есть **stand-down** — лучше не котироваться, чем котироваться по протухшему
якорю.

In [9]:
def quotes(micro_L, age, basis, q, p):
    "quote-функция. p - dict параметров. Возвращает (bid, ask) или (None,None) при stale."
    if age > p["max_age"]:
        return None, None
    half = 1.0/p["k"] + p["hedge_cost"] + p["adverse"] + p["age_slope"]*age
    anchor_base = micro_L - basis                  # basis: якорная биржа дороже нашей -> опускаем якорь
    skew = -p["c"]*q                       # длинная позиция опускает котировки — хотим продать
    return anchor_base + skew - half, anchor_base + skew + half

def hedge_qty(q, theta):
    "hedge-правило: закрываем позицию целиком, только если она вышла за порог"
    return -q if abs(q) > theta else 0.0

print("чистые функции определены: quotes(micro_L, age, basis, q, p), hedge_qty(q, theta)")
print("пример: quotes(100.0, 1.0, 0.0, 3.0, P0) ->", quotes(100.0, 1.0, 0.0, 3.0, dict(k=1.5, hedge_cost=0.0085, adverse=0.0, age_slope=2.0, c=0.05, max_age=5)))

чистые функции определены: quotes(micro_L, age, basis, q, p), hedge_qty(q, theta)
пример: quotes(100.0, 1.0, 0.0, 3.0, P0) -> (97.17483333333332, 102.52516666666666)


Проверяем чистые функции unit-тестами: они не должны зависеть от истории, должны быть симметричны по
позиции, монотонны по age и давать `None` при протухании.

In [10]:
P0 = dict(k=1.5, hedge_cost=0.0085, adverse=0.0, age_slope=2.0, c=0.05, max_age=5)  # bp за секунду молчания якоря

# 1. skew по позиции: q и -q дают зеркальные центры вокруг якоря, спред одинаков
b1,a1 = quotes(100.0, 0.5, 0.0, +2.0, P0)
b2,a2 = quotes(100.0, 0.5, 0.0, -2.0, P0)
assert abs(((b1+a1)/2 - 100.0) + ((b2+a2)/2 - 100.0)) < 1e-12, "skew не зеркален по q"
assert abs((a1-b1) - (a2-b2)) < 1e-12, "spread зависит от q"
assert abs((a2-a1) - P0["c"]*4.0) < 1e-12, "skew по q неверен (q>0 должен опускать котировки)"

# 2. age расширяет спред, центры совпадают
b3,a3 = quotes(100.0, 2.0, 0.0, 0.0, P0)
b4,a4 = quotes(100.0, 0.0, 0.0, 0.0, P0)
assert (a3-b3) > (a4-b4), "age не расширяет спред"
assert abs((b3+a3) - (b4+a4)) < 1e-12, "age сдвинул центр"

# 3. stale -> None
assert quotes(100.0, P0["max_age"]+0.001, 0.0, 0.0, P0) == (None,None), "нет stand-down"

# 4. basis опускает якорь ровно на basis
b5,a5 = quotes(100.0, 0.0, 0.5, 0.0, P0)
assert abs((b5+a5)/2 - (100.0-0.5)) < 1e-12, "basis не учтён"

# 5. hedge_qty
assert hedge_qty(0.5, 1.0) == 0.0 and hedge_qty(2.0, 1.0) == -2.0

# 6. purity: те же входы -> те же выходы (нет скрытого состояния)
assert quotes(100.0,0.3,0.1,1.0,P0) == quotes(100.0,0.3,0.1,1.0,P0), "не чистая функция"

# 7. независимость от порядка вызовов
_ = [quotes(100.0+i, 0.1, 0.0, 0.0, P0) for i in range(100)]
assert quotes(100.0,0.3,0.1,1.0,P0) == quotes(100.0,0.3,0.1,1.0,P0)
print("unit-тесты движка: 7/7 ok")
print("пример: quotes(micro_L=100, age=1s, basis=0, q=+3):", tuple(round(x,4) for x in quotes(100.0, 1.0, 0.0, 3.0, P0)))

unit-тесты движка: 7/7 ok
пример: quotes(micro_L=100, age=1s, basis=0, q=+3): (97.1748, 102.5252)


Все семь тестов прошли. Пример: при $q=+3$, $c=0.05$ центр уходит **вниз** на $0.15$ bp — длинную позицию
разгружаем тем, что опускаем обе котировки: ask подтягивается к рынку и чаще исполняется, bid уходит и не
докупает. Спред при этом не зависит от $q$ и держится на $1/k+h+\lambda\cdot\text{age}$.

Теперь оценщик basis с заявленным half-life и break detector. Basis — медленная величина, её правильно
сглаживать EWMA с контролируемой памятью: half-life $T_{1/2}$ задаёт коэффициент
$\alpha = 1-\exp(-\ln 2\cdot\Delta t/T_{1/2})$ на шаге $\Delta t$. Чем больше $T_{1/2}$, тем стабильнее оценка,
но тем медленнее реакция на скачок — классический выбор «шум против запаздывания». Break detector — триггер:
если модуль приращения basis превысил порог, сбрасываем оценку и уходим в stand-down.

In [11]:
def ewma_alpha(half_life_s, dt_s):
    "коэффициент EWMA для заданного half-life"
    return 1.0 - np.exp(-np.log(2.0)*dt_s/half_life_s)

def basis_estimator(basis_series, half_life_s, dt_s=1.0, thresh=2.0):
    "EWMA-оценка basis + список индексов-скачков (break detector)"
    a = ewma_alpha(half_life_s, dt_s)
    est = np.empty(len(basis_series)); est[0] = basis_series[0]
    breaks = []
    for i in range(1, len(basis_series)):
        est[i] = est[i-1] + a*(basis_series[i]-est[i-1])
        if abs(basis_series[i]-basis_series[i-1]) > thresh:
            breaks.append(i)
    return est, breaks

b1s = basis["Binance"].resample("1s").last().dropna()          # 1s сетка для half-life в секундах
print("EWMA-оценка basis (half-life 5/20/60/300 s): sd ошибки и ошибка через 10 с после скачка 2 bp")
seg = b1s.iloc[5000:5600].copy(); seg.iloc[100:] += 2.0       # искусственный скачок 2 bp
for hl_ in [5,20,60,300]:
    est, _ = basis_estimator(b1s.to_numpy(), hl_)
    e2, _  = basis_estimator(seg.to_numpy(), hl_)
    err10 = abs(e2[110]-seg.to_numpy()[110])
    print("  half-life", str(hl_).rjust(3), "s:  sd(est)=%.4f | sd(est-basis)=%.4f bp | ошибка через 10 с = %.3f bp" % (np.std(est), np.std(est-b1s.to_numpy()), err10))

est5, br5 = basis_estimator(b1s.to_numpy(), 5)
print("break detector (thresh=2 bp) нашёл скачков:", len(br5), "на", len(b1s), "секундах")

EWMA-оценка basis (half-life 5/20/60/300 s): sd ошибки и ошибка через 10 с после скачка 2 bp
  half-life   5 s:  sd(est)=0.7771 | sd(est-basis)=0.3030 bp | ошибка через 10 с = 0.121 bp


  half-life  20 s:  sd(est)=0.7533 | sd(est-basis)=0.3701 bp | ошибка через 10 с = 0.869 bp
  half-life  60 s:  sd(est)=0.7318 | sd(est-basis)=0.4141 bp | ошибка через 10 с = 1.319 bp


  half-life 300 s:  sd(est)=0.6776 | sd(est-basis)=0.4971 bp | ошибка через 10 с = 1.498 bp
break detector (thresh=2 bp) нашёл скачков: 322 на 172643 секундах


Компромисс виден, но на 1s-сетке он устроен не так, как в лабораторной таблице слайда: basis там «дышит»
высокочастотно, поэтому короткая память (5 с) ошибается против самой серии меньше всех ($0.30$ bp) и быстрее
всех догоняет скачок (через 10 с ошибка $0.12$ bp), а длинная (300 с) сглаживает оценку, но отстаёт от дыхания
($0.50$ bp) и после скачка сидит на $1.5$ bp — «не замечает» сдвиг и торгует по протухшему basis. У слайдных
$0.130\to0.014$ bp метрикой была гладкость самой оценки — наш $sd$ оценки падает с $0.78$ до $0.68$ bp (колонка
sd(est)), тогда как $sd$ ошибки против серии растёт с $0.30$ до $0.50$; направление компромисса то же. Отсюда смысл «заяви half-life»: у каждой настройки своя цена — быстрые рискуют false break,
медленные — stale basis. Break detector в паре с этим: как только приращение basis превысило порог, EWMA-оценку
надо сбросить и уйти в stand-down, а не продолжать усреднять скачок.

Оговорка по роли: этот оценщик — демонстрация EWMA и break detector, которую требует пункт 2. В симуляторе
пункта 3 basis не оценивается динамически (там упрощённая quote-функция без basis), а скачок basis задаётся
сценарно в пункте 5.

## Пункт 3. Simulate — филлы явной моделью, якоря, latency

Симулятор ставит котировки по политике, тянет филлы **независимыми Бернулли** по сторонам и ведёт позицию.
Главная грабля (её ловили в 11-м дне): если сделать филлы взаимоисключающими (bid только если ask не сработал),
inventory тихо уезжает; тест симметрии это поймает. Вероятность филла за шаг — это пуассоновская интенсивность
$\lambda(\delta)=Ae^{-k\delta}$, переведённая в вероятность $p=\lambda\,\Delta t$; по сторонам филлы
разыгрываются независимо.

Две оговорки к движку. Первая: симулятор использует **упрощённую** quote-функцию $\text{half}=1/k+h$ — без
skew, age и basis. Члены $c\,q$, $\lambda\,\text{age}$ и basis из `quotes()` показаны отдельно (unit-тесты,
stale-таблица и сценарии break-it) и на симуляционные числа этой секции не влияют. Вторая: поскольку разведка
не различила лидера, якорь на Bybit в таблице ниже — выбранная fair-цена модели, а не выведенное из данных
лидерство; сравнение якорей измеряет механику движка (нулевая ошибка якоря против ошибки $e$).

Юнит-экономика одного хеджированного филла (слайд 6):

$$\text{P\&L per fill} = \delta + (\text{anchor}-\text{true}) - h - a,\qquad
\text{P\&L/hour} = 2Ae^{-k\delta}(\delta-h)\cdot 3600.$$

Оптимум по $\delta$ — это $\delta^* = 1/k + h$: из максимизации $Ae^{-k\delta}(\delta-h)$ производная даёт
$-k(\delta-h)+1=0$. Сначала проверим closed form, потом симулятор.

In [12]:
A0, K0, H0 = 3.0, 1.5, 0.5
def plh_hour(delta, A=A0, k=K0, h=H0):
    "ожидаемый P&L/hour одного хеджированного филла как функция half-spread"
    return 2*A*np.exp(-k*delta)*(delta-h)*3600

d_star = 1/K0 + H0
print("closed form: A=%s, k=%s, h=%s" % (A0, K0, H0))
print("  d* = 1/k + h = %.4f + %s = %.4f bp  ->  P&L/hour = %.0f" % (1/K0, H0, d_star, plh_hour(d_star)))
print("  при 1/k=%.4f (спред без поправки на h, тот же h=0.5 в издержке): P&L/hour = %.0f" % (1/K0, plh_hour(1/K0)))
print("  при h=0 (бесплатный хедж, d*=1/k): P&L/hour = %.0f" % plh_hour(1/K0, h=0.0))
ds = np.linspace(0.2, 3.0, 400); vv = plh_hour(ds)
print("  численный argmax = %.4f bp (P&L/hour %.0f)" % (ds[int(np.argmax(vv))], vv.max()))

def symmetry(A, k, delta, dt, T=600, W=4000, seed=0):
    rng = np.random.default_rng(seed); q = np.zeros(W); acc = 0.0
    pb = min(A*np.exp(-k*delta)*dt, 1.0)
    for _ in range(T):
        fb = (rng.random(W) < pb).astype(float); fa = (rng.random(W) < pb).astype(float)
        acc += np.mean(q*fb - q*fa); q = q + fb - fa
    return acc/T
print("\nтест симметрии mean(q_before*side) при lambda*dt=%.2f: %+.5f (ожидание 0)" % (A0*np.exp(-K0*d_star)*0.1, symmetry(A0,K0,d_star,0.1)))

closed form: A=3.0, k=1.5, h=0.5
  d* = 1/k + h = 0.6667 + 0.5 = 1.1667 bp  ->  P&L/hour = 2502
  при 1/k=0.6667 (спред без поправки на h, тот же h=0.5 в издержке): P&L/hour = 1324
  при h=0 (бесплатный хедж, d*=1/k): P&L/hour = 5297
  численный argmax = 1.1684 bp (P&L/hour 2502)

тест симметрии mean(q_before*side) при lambda*dt=0.05: +0.00027 (ожидание 0)


Closed form сошёлся: $\delta^*=1.1667$ bp даёт $2502$/час — это ровно число слайда, и иначе быть не могло:
та же формула при тех же $(A,k,h)$, что в примере слайда; это алгебра, а не воспроизведение. Дальше идут прогоны
на реальных путях, и их абсолютные числа с лабораторной таблицей слайда ($1773$/$2564$/$2487$) не совпадают
по построению. Отдельно видна цена «забытого хеджа»: если котироваться по субоптимальному спреду
$\delta=1/k$ — не заложив $h$ в уровень, — P&L падает с $2502$ до $1324$; хедж-кост входит в решение дважды
(и в оптимум, и в кривую), и его игнорирование стоит почти вдвое. Это не то же самое, что $h=0$: при бесплатном хедже
оптимум и есть $1/k$, и P&L там *выше* ($5297$) — теряет не отмена хедж-коста, а ошибка в уровне спреда. Тест
симметрии на независимых Бернулли даёт дрейф позиции $\approx 0$ — взаимоисключающего бага нет.

Теперь симулятор на **реальных траекториях**. Роли: quote venue $M$ — Binance, якорь $L$ — Bybit (в модели
fair = Bybit; разведка не различила лидера, так что это иллюстративный выбор точки отсчёта, а не вывод из
корреляций), hedge venue — Bybit/Okex. Траектории — реальные mid/microprice с 100-мс сетки, каждое окно 60 с —
одна независимая траектория. P&L считаем как сумму трёх частей: **spread capture** каждого филла против
истинной цены якорной биржи, минус **стоимость хеджа** $h$ за каждую единицу, плюс **mark инвентаря** — позиция,
которую мы держим между хеджами, переоценивается по той же истинной цене (эта переоценка может быть и в плюс,
и в минус — знак несёт риск, а не вычитаемая константа). Latency моделируем задержкой якоря:
на шаге $t$ движок видит цену якорной биржи, снятую $L$ шагов назад.

In [13]:
def windows(x, n_win, win):
    x = np.asarray(x)[:n_win*win]
    return x.reshape(n_win, win)

NW, WIN = 300, 601                     # 300 окон по 60 с на 100ms сетке
mL = (mid["Bybit"]   - mid["Bybit"].iloc[0])   / mid["Bybit"].iloc[0]   * bp
mM = (mid["Binance"] - mid["Binance"].iloc[0]) / mid["Binance"].iloc[0] * bp
mI = (miP["Bybit"]   - miP["Bybit"].iloc[0])   / miP["Bybit"].iloc[0]   * bp
WL, WM, WI = windows(mL,NW,WIN), windows(mM,NW,WIN), windows(mI,NW,WIN)
print("окон:", NW, "| шагов в окне:", WIN, "| dt=0.1s -> горизонт %.0f с" % (WIN*0.1))
print("sd приращения mid якорной биржи (Bybit) за 100ms, bp: %.4f" % np.std(np.diff(WL,axis=1)))

from collections import deque
def sim_pl(Mm, fair, kind, lag, policy, theta, p, delay=1, mic=None,
           anchor_src=None, active=None, seed=0, dt=0.1):
    """Котируемся на Mm вокруг якоря; fair - истинная цена (якорная биржа Bybit).
    anchor_src - источник якоря, если он отличается от fair (напр. протухший якорь).
    pl = spread capture - стоимость хеджа + mark инвентаря по fair. Возвращает (pl, филлы, хеджировано, q, (sp,hc,ir)).
    active - вектор по шагам: 0 = stand-down (не котируемся), None = всегда котируемся."""
    rng = np.random.default_rng(seed); nw, T = Mm.shape
    asrc = fair if anchor_src is None else anchor_src
    q = np.zeros(nw); sp = np.zeros(nw); hc = np.zeros(nw); ir = np.zeros(nw)
    nfill = np.zeros(nw); hedged = np.zeros(nw); pend = np.zeros(nw); queue = deque()
    for t in range(T-1):
        j = max(t-lag, 0)
        if kind == "own":          anc = Mm[:,j]
        elif kind == "lead_micro": anc = (mic if mic is not None else asrc)[:,j]
        else:                      anc = asrc[:,j]
        half = 1.0/p["k"] + p["hedge_cost"]   # упрощённая quote-функция: без skew/age/basis (оговорка выше)
        bid, ask = anc-half, anc+half; m = Mm[:,t]
        gate = 1.0 if active is None else active[t]         # stand-down выключает обе стороны
        pb = np.clip(p["A"]*np.exp(-p["k"]*(m-bid))*dt, 0, 1)*gate
        pa = np.clip(p["A"]*np.exp(-p["k"]*(ask-m))*dt, 0, 1)*gate
        fb = (rng.random(nw) < pb).astype(float)          # НЕЗАВИСИМО
        fa = (rng.random(nw) < pa).astype(float)
        sp += fb*(fair[:,t]-bid) + fa*(ask-fair[:,t])     # spread capture против истинной цены
        q += fb - fa; nfill += fb + fa
        target = -q if policy == "every" else np.where(np.abs(q) > theta, -q, pend)
        order = target - pend
        if delay == 0:
            hq = order
        else:
            queue.append(order); hq = queue.popleft() if len(queue) > delay else np.zeros(nw)
        pend += order - hq
        ir += q*(fair[:,t+1]-fair[:,t])                   # mark инвентаря за шаг
        q += hq; hc += np.abs(hq)*p["hedge_cost"]; hedged += np.abs(hq)
    ir += q*(fair[:,-1]-fair[:,-2]); hc += np.abs(q)*p["hedge_cost"]
    return sp - hc + ir, nfill, hedged, q, (sp, hc, ir)

P_SIM = dict(A=3.0, k=1.5, hedge_cost=0.5)               # h=0.5 bp как на слайде
print("\nякоря и latency: P&L за окно 60 с, bp (policy threshold th=2, hedge delay 100ms)")
tab = []
for lag in [0,1,2,5]:
    row = {"latency_ms": lag*100}
    row["own mid"]      = round(sim_pl(WM, WL, "own",        lag, "threshold", 2, P_SIM)[0].mean(), 0)
    row["Bybit mid"]    = round(sim_pl(WM, WL, "lead",       lag, "threshold", 2, P_SIM)[0].mean(), 0)
    row["Bybit micro"]  = round(sim_pl(WM, WL, "lead_micro", lag, "threshold", 2, P_SIM, mic=WI)[0].mean(), 0)
    tab.append(row)
print(pd.DataFrame(tab).to_string(index=False))

окон: 300 | шагов в окне: 601 | dt=0.1s -> горизонт 60 с
sd приращения mid якорной биржи (Bybit) за 100ms, bp: 0.2510

якоря и latency: P&L за окно 60 с, bp (policy threshold th=2, hedge delay 100ms)


 latency_ms  own mid  Bybit mid  Bybit micro
          0     64.0      115.0        116.0
        100     56.0      105.0        105.0
        200     47.0       93.0         94.0
        500     15.0       60.0         60.0


Таблица воспроизводит качественную картину слайда: **якорь с нулевой ошибкой — крупный эффект**. Оговорка
из разведки в силе: «лидерский» якорь — это mid Bybit, fair-цена модели, а не выведенное из данных лидерство;
сравнение измеряет механику движка (якорь $=$ fair против якоря с ошибкой $e$). На нулевой latency якорь
на своём mid даёт $\approx 64$ bp за окно против $\approx 115$ у якоря на Bybit, а с ростом latency собственный
mid быстро разваливается: на 500 мс он падает до $\approx 15$, тогда как якорь на Bybit держится на $\approx 60$ —
разрыв вчетверо. Причина структурная: ошибка якоря $e=\text{fair}-\text{anchor}$ входит в P&L, и на своём mid
она растёт с latency, а на якорной бирже равна нулю по построению.

Microprice добавляет к якорю на Bybit мало: $116$ против $115$ на нулевой задержке, $94$ против $93$ на
200 мс. На этих окнах разница в пределах шума, потому что сигнал microprice живёт 1–2 такта и в среднем по
60-секундному окну размывается базисом и спредом. **Якорь, совпадающий с fair, — главный источник эджа,
microprice — второй порядок** и проявляется только на согласованном горизонте. Ниже покажем связь с hedge
delay. Абсолютные числа этой таблицы (64/115 bp за окно) — наши, а не лабораторные
$1773$/$2564$ из слайда; совпадает только порядок и направление эффектов.

Прогоняем sweep по $A$ и $k$: как P&L реагирует на допущения об интенсивности филлов. Это ещё раз показывает,
насколько числа условны.

In [14]:
print("sweep A x k: P&L за окно 60 с, bp (якорь Bybit mid, latency 100 мс)")
piv = pd.DataFrame(index=["A="+str(a) for a in [2,5,10]], columns=["k="+str(kk) for kk in [1.0,1.5,3.0]], dtype=float)
for a in [2,5,10]:
    for kk in [1.0,1.5,3.0]:
        P = dict(A=a, k=kk, hedge_cost=0.5)
        piv.loc["A="+str(a), "k="+str(kk)] = round(sim_pl(WM, WL, "lead", 1, "threshold", 2, P)[0].mean(), 0)
print(piv.to_string())

sweep A x k: P&L за окно 60 с, bp (якорь Bybit mid, latency 100 мс)


      k=1.0  k=1.5  k=3.0
A=2    93.0   73.0   47.0
A=5   230.0  163.0   75.0
A=10  431.0  269.0  105.0


P&L растёт по $A$ (интенсивность филлов) и падает по $k$ (оптимальный спред шире, филлов меньше): при
$A=2$ числа скромные, при $A=10$ — в разы больше. Всё это диапазон допущений, а не измерений. Мы фиксируем
эти числа как **условные** и переходим к хеджу, где можно проверить больше.

### Пять тестов симулятора

Пять проверок, отсекающих типовые ошибки: симметрия филлов, оптимум closed form, стоимость хеджа по ручному
проходу книги, oracle-якорь и отсутствие look-ahead («при возмущении будущего котировки на текущем шаге не
меняются»).

In [15]:
print("=== 1. fill symmetry ===")
print("  mean(q_before*side), lambda*dt=%.2f: %+.5f  (0 = ок)" % (3.0*np.exp(-1.5*1.167)*0.1, symmetry(3.0,1.5,1.167,0.1)))

print("\n=== 2. hedged optimum = 1/k + h ===")
grid = np.linspace(0.4, 2.5, 200)
best = grid[int(np.argmax(plh_hour(grid)))]
print("  argmax P&L/hour = %.4f bp | 1/k+h = %.4f bp" % (best, d_star))

print("\n=== 3. self-consistency: walk(4) против полспреда той же книги ===")
bk, m = BOOKS["Bybit"], MIDS["Bybit"]
manual = walk(bk, -1, 4, m)                          # 4 units по лучшему уровню
print("  walk 4 units: %.4f bp | лучшие ask-цены: %s" % (manual, list(bk[bk.side==-1].sort_values("price").price.head(4))))
print("  полспреда = tick/2 = %.4f bp -> 4 units должны стоить ровно полспреда: %.4f bp" % (TICK/m*1e4/2, TICK/m*1e4/2))

print("\n=== 4. oracle anchor ===")
pl_or,_,_,_,_ = sim_pl(WM, WL, "lead", 0, "threshold", 2, P_SIM)
pl_im,_,_,_,_ = sim_pl(WM, WL, "lead", 5, "threshold", 2, P_SIM)     # протухший якорь
print("  P&L oracle(lat=0)=%.0f >= impaired(lat=500ms)=%.0f: %s" % (pl_or.mean(), pl_im.mean(), pl_or.mean() >= pl_im.mean()))

print("\n=== 5. timing: котировки на t не зависят от будущего ===")
def quotes_at(Mmt, tstop, lag, p):
    j = max(tstop-lag,0); anc = Mmt[0,j]; half = 1/p["k"]+p["hedge_cost"]
    return float(anc-half), float(anc+half)
A_ = quotes_at(WM, 100, 1, P_SIM)
B_ = quotes_at(WM[:, :101], 100, 1, P_SIM)
Mp = WM.copy(); Mp[:, 101:] += 100.0
C_ = quotes_at(Mp, 100, 1, P_SIM)
print("  prefix: котировки неизменны: %s;  perturbation: %s" % (A_ == B_, A_ == C_))

=== 1. fill symmetry ===
  mean(q_before*side), lambda*dt=0.05: +0.00028  (0 = ок)

=== 2. hedged optimum = 1/k + h ===
  argmax P&L/hour = 1.1704 bp | 1/k+h = 1.1667 bp

=== 3. self-consistency: walk(4) против полспреда той же книги ===
  walk 4 units: 0.0085 bp | лучшие ask-цены: [58706.4, 58706.5, 58706.7, 58706.8]
  полспреда = tick/2 = 0.0085 bp -> 4 units должны стоить ровно полспреда: 0.0085 bp

=== 4. oracle anchor ===
  P&L oracle(lat=0)=115 >= impaired(lat=500ms)=60: True

=== 5. timing: котировки на t не зависят от будущего ===
  prefix: котировки неизменны: True;  perturbation: True


Все пять тестов пройдены — вместе с семью unit-тестами чистых функций из пункта 2 это 12 проверок.
Симметрия даёт дрейф $0$ на независимых Бернулли. Оптимум совпал с
$1/k+h=1.1667$ bp в пределах шага сетки ($0.01$ bp) — точнее сетка из 200 точек не различает. Третья проверка —
self-consistency: $4$ units по книге стоят ровно полспреда платформы, и это сравнение walk'а с walk'ом по той
же книге — контроль реализации, а не независимое подтверждение. Oracle-якорь даёт
P&L больше протухшего. Котировки на шаге $t$ не зависят ни от усечения, ни от возмущения будущего — это самый
прямой тест на отсутствие look-ahead, и он обеспечен конструкцией `quotes`: функция видит только текущий якорь
и параметры. Отдельно о знаке mark: в P&L шаг $t$ маркируется по $fair_{t+1}$ — это переоценка уже принятого
решения, а не вход в него; котировки на $t$ построены только по данным до $t$ включительно.

### Горизонт microprice должен совпадать с hedge delay

Якорь из microprice полезен только на том горизонте, на котором microprice предсказывает. Если хедж
запаздывает на $\tau$, маркировать надо по цене через тот же $\tau$ — поэтому сравниваем якоря по mid и по microprice,
когда и якорь, и mark используют горизонт $\tau$.

In [16]:
print("горизонт microprice vs hedge delay (якорь в t, mark по fair[t+tau] — хедж доезжает через tau):")
# филл, принятый по якорю в t, маркируется ценой через tau: та же логика, что незакрытая позиция
# живёт tau секунд. Сравниваем mid- и micro-якорь на согласованном горизонте.
def sim_horizon(d):
    nw, T = WM.shape
    res = {}
    for kind, mic in [("lead", None), ("lead_micro", WI)]:
        q = np.zeros(nw); sp = np.zeros(nw); hc = np.zeros(nw)
        rng = np.random.default_rng(0)
        p = P_SIM
        for t in range(T-1-d):
            anc = (mic if mic is not None else WL)[:, t]
            half = 1/p["k"]+p["hedge_cost"]; bid, ask = anc-half, anc+half; m = WM[:,t]
            pb = np.clip(p["A"]*np.exp(-p["k"]*(m-bid))*0.1,0,1); pa = np.clip(p["A"]*np.exp(-p["k"]*(ask-m))*0.1,0,1)
            fb = (rng.random(nw)<pb).astype(float); fa = (rng.random(nw)<pa).astype(float)
            fairmark = WL[:, t+d]
            sp += fb*(fairmark-bid) + fa*(ask-fairmark); q += fb-fa
            hq = np.where(np.abs(q)>2, -q, 0.0); q += hq; hc += np.abs(hq)*p["hedge_cost"]
        res[kind] = (sp-hc).mean()
    return res
for d, name in [(0,"0–100 ms"),(5,"500 ms"),(20,"2000 ms")]:
    r = sim_horizon(d)
    print("  горизонт %8s: mid=%8.0f  micro=%8.0f" % (name, r["lead"], r["lead_micro"]))

горизонт microprice vs hedge delay (якорь в t, mark по fair[t+tau] — хедж доезжает через tau):
  горизонт 0–100 ms: mid=     125  micro=     125


  горизонт   500 ms: mid=     113  micro=     113
  горизонт  2000 ms: mid=     105  micro=     106


Во всех трёх случаях mid и micro идут вплотную ($125$ против $125$ на нулевой задержке, $113$ против
$113$ на 500 мс, $105$ против $106$ на 2 с). На этих окнах microprice почти не добавляет к mid, потому что его
сигнал живёт 1–2 такта и в среднем по окну размывается. Полезный вывод здесь в том, что
**горизонт якоря надо согласовывать с горизонтом хеджа**: если хедж запаздывает на $\tau$, то и якорь, и mark
живут на $\tau$, и на длинном $\tau$ mark несёт волатильность $\sigma\sqrt{\tau}$, которая съедает спред
(P&L падает с $125$ до $105$ при переходе от нулевой задержки хеджа к 2 с). Несовпадение горизонта — это цена рассогласования
из слайда 12; в наших данных microprice даёт лишь небольшой вклад поверх этого. Лабораторные $-55\%$ слайда —
иллюстрация на других $(A,k,\beta,\sigma)$: на реальных путях разница mid/micro измеряется десятыми долями процента,
и мы не подаём её как воспроизведение слайда.

## Пункт 4. Hedge — every-fill, threshold, certainty equivalent, hedge delay

Хедж ограничивает риск, но каждая единица платит $h$ bp. Отсюда две чистые политики: **every-fill**
(весь накопленный объём в хедж-венью на каждом филле) и **threshold** (ждём, пока позиция не выйдет за
$\theta$, и только тогда хеджируем целиком). Threshold берёт на себя больше позиционного риска, но платит $h$
реже — и обычно выигрывает, потому что филлы по сторонам в значительной мере взаимно гасятся.

Критерий сравнения — **certainty equivalent** (CE) экспоненциальной полезности:
$\text{CE} = -\frac1\gamma\ln\mathbb E[e^{-\gamma X}]$. Он штрафует дисперсию: «много филлов, много хеджа»
и «мало филлов, много риска» сравниваются в одной шкале. Меряем CE на P&L за окно 60 с — в тех же bp, в
которых живёт вся задача. Чтобы $e^{-\gamma X}$ не взрывался на хвостах, P&L окна ограничиваем $\pm 6/\gamma$ bp
(при $\gamma=0.10$ отсечка $\pm 60$ bp активна и давит CE вверх — колонку $\gamma=0.10$ читаем с этим запасом).

Хедж исполняется не мгновенно, а через $\tau$ шагов; всё это время позиция несёт риск движения mid,
который идёт как $\sigma\sqrt{\tau}$. Сравниваем политики при трёх риск-аппетитах.

In [17]:
def ce(x, g, clip=6.0):
    c = clip/g
    return -(1.0/g)*np.log(np.mean(np.exp(-g*np.clip(x, -c, c))))

print("hedge policy: CE (bp за окно 60с) по риск-аппетиту gamma, якорь Bybit mid (lag=100ms)")
print("%6s | " % "gamma" + " ".join("th=%-3d" % th for th in [0,1,2,4,8]) + "  (th=0 ~ every-fill)")
for gm in [0.01, 0.03, 0.10]:
    row = []
    for th in [0,1,2,4,8]:
        pol = "every" if th == 0 else "threshold"
        pl, _, _, _, _ = sim_pl(WM, WL, "lead", 1, pol, th, P_SIM)
        row.append(ce(pl, gm))
    b = int(np.argmax(row))
    print("%6.2f | " % gm + " ".join("%7.2f" % v for v in row) + "   -> оптимум th=%d" % [0,1,2,4,8][b])

print("\nразложение P&L (bp за окно): spread capture, стоимость хеджа, mark инвентаря")
print("%10s %8s %9s %9s %7s" % ("policy","pl","spread","hedge_cost","irisk"))
for pol, th in [("every",0),("threshold",2),("threshold",4),("threshold",8)]:
    _, _, _, _, (SP, HC, IR) = sim_pl(WM, WL, "lead", 1, pol, th, P_SIM)
    print("%10s %8.1f %9.1f %9.1f %7.1f" % (pol+str(th), (SP-HC+IR).mean(), SP.mean(), HC.mean(), IR.mean()))

hedge policy: CE (bp за окно 60с) по риск-аппетиту gamma, якорь Bybit mid (lag=100ms)
 gamma | th=0   th=1   th=2   th=4   th=8    (th=0 ~ every-fill)


  0.01 |   64.88   77.09   78.70   80.24   79.55   -> оптимум th=4


  0.03 |    3.66   25.49   25.76   31.76   21.89   -> оптимум th=4


  0.10 |  -12.25  -12.23  -15.40  -10.08  -15.62   -> оптимум th=4

разложение P&L (bp за окно): spread capture, стоимость хеджа, mark инвентаря
    policy       pl    spread hedge_cost   irisk
    every0     93.3     187.3      83.4   -10.6
threshold2    104.6     187.3      71.1   -11.6


threshold4    105.9     187.3      69.0   -12.5
threshold8    109.0     187.3      67.6   -10.7


Главное в таблице совпало с ожиданием: **по среднему P&L и при малых $\gamma$ threshold не хуже
every-fill** (в разложении ниже — 93 bp против 105–109). А ожидание слайда про сдвиг оптимума к меньшему порогу
с ростом $\gamma$ на наших траекториях не подтвердилось: при всех трёх $\gamma$ лучше $\theta=4$, причём колонка
$\gamma=0.10$ задавлена отсечкой CE ($\pm60$ bp) и различия порогов в ней ($-10\ldots-16$) — шум. Это ровно тот
случай «результат, не выживший в sweep, — находка»: зависимость оптимума от риск-аппетита существует, но на
наших 300 окнах она не различима. Every-fill («$\theta=0$») при этом не лучший ни при одном $\gamma$: он платит $h$ на каждом филле, в том числе на взаимно гасящихся, и потому проигрывает
пороговой политике. Разложение показывает, откуда разница: spread capture у всех политик одинаков, а стоимость
хеджа у threshold ниже — именно на этом он и выигрывает.

Теперь sweep hedge delay. Незакрытая позиция несёт риск движения mid за $\tau$ секунд размером
$\sigma\sqrt{\tau}$; для медианной $\sigma$ Bybit это даёт доли bp при малых $\tau$ и больше — при
больших. Смотрим, как от задержки зависит CE.

In [18]:
sig = np.median(np.std(np.diff(WL,axis=1),axis=1))/np.sqrt(0.1)     # bp/sqrt(s)
print("median sigma(Bybit) ~ %.3f bp/sqrt(s)" % sig)
print("незахеджированное окно sqrt(tau)*sigma:")
for tau_ms in [50, 500, 2000]:
    print("  tau=%4d ms: %.2f bp" % (tau_ms, sig*np.sqrt(tau_ms/1000)))

print("\nCE vs hedge delay (threshold th=2, gamma=0.03) — среднее по 8 сидам:")
for delay in [0,1,3,5,10]:
    ces, mns, sds = [], [], []
    for sd_ in range(8):
        pl, _, hg, _, _ = sim_pl(WM, WL, "lead", 1, "threshold", 2, P_SIM, delay=delay, seed=sd_)
        ces.append(ce(pl, 0.03)); mns.append(pl.mean()); sds.append(pl.std())
    print("  hedge delay %4d ms: CE=%8.2f  pl mean=%7.2f sd=%6.2f" % (delay*100, np.mean(ces), np.mean(mns), np.mean(sds)))

median sigma(Bybit) ~ 0.530 bp/sqrt(s)
незахеджированное окно sqrt(tau)*sigma:
  tau=  50 ms: 0.12 bp
  tau= 500 ms: 0.37 bp
  tau=2000 ms: 0.75 bp

CE vs hedge delay (threshold th=2, gamma=0.03) — среднее по 8 сидам:


  hedge delay    0 ms: CE=   40.00  pl mean= 106.76 sd= 81.35


  hedge delay  100 ms: CE=   19.41  pl mean= 104.79 sd= 81.69


  hedge delay  300 ms: CE=   15.86  pl mean= 102.65 sd= 82.02


  hedge delay  500 ms: CE=   35.12  pl mean= 100.89 sd= 81.37


  hedge delay 1000 ms: CE=   28.91  pl mean=  96.72 sd= 81.96


Риск незахеджированного окна растёт как корень из задержки — тот же порядок, что предсказан из
$\sigma\sqrt{\tau}$ ($0.12$ bp на 50 мс, $0.75$ bp на 2 с). Лабораторный эталон слайда ($0.22$/$0.71$/$1.41$ bp) выше
примерно вдвое на малых задержках: у нас медианная волатильность Bybit за такт оказалась меньше, чем в
иллюстрации слайда, — величина $\sigma$ измерена, а не взята из легенды. Средний P&L монотонно падает с задержкой
($107 \to 97$ bp), дисперсия держится около $81$ bp. Сама кривая CE при этом немонотонна (на 500 мс она выше,
чем на 300 мс): пять точек задержки — мало, чтобы выводить форму CE-кривой, отдельные шаги накрыты шумом по
300 окнам (усреднение по 8 сидам сглаживает, но не убирает); тренд читаем только по среднему P&L. Вывод
качественный и устойчивый: быстрый
хедж важен не сам по себе, а потому что **укорачивает окно, в котором мы держим риск** — и поэтому в пункте 3
горизонт якоря надо было согласовать с этой задержкой.

## Пункт 5. Break it — basis jump, stale leader, thin hedge book

Модель ломается тремя способами, и каждый проверяется ровно тем механизмом, который для него построен.

**Basis jump.** Если basis мгновенно сдвигается на $b$, а мы котируемся по старому якорю, эффективное
расстояние котировки до справедливой цены становится $\delta \pm b$, и интенсивность филла меняется в
$e^{\mp kb}$ — сторона, оказавшаяся «ближе» к рынку, начинает срабатывать в разы чаще. При $b=1$ bp и
$k=1.5$ множитель $e^{1.5}=4.5$; при $b=2$ bp — уже $20$, и мы неконтролируемо набираем позицию. Это ловит
break detector из пункта 2: скачок выше порога — сброс оценки и stand-down.

**Stale leader.** Если лидер молчит, котироваться по протухшему якорю опасно: с ростом age мы не знаем, где
рынок. Правильная реакция — расширять спред, а за максимальным age **остановиться, а не откатываться на
собственный mid** (свой mid при latency сам протух, см. пункт 3).

In [19]:
print("=== break it: basis jump -> fill-rate ===")
for kk in [1.0, 1.5, 3.0]:
    print("  k=%.1f: множитель fill-rate exp(k*b) при b=1 bp = %.2f" % (kk, np.exp(kk*1.0)))
print("  jump 2 bp при k=1.5: множитель exp(3)=%.1f" % np.exp(1.5*2))

print("\n=== break it: stale leader -> расширение спреда и stand-down ===")
print("  age (ms) | half-spread (bp) | режим")
for age in [0.0, 0.1, 0.25, 0.5, 1.0, P0["max_age"]+0.01]:
    b,a = quotes(100.0, age, 0.0, 0.0, P0)
    if b is None: print("  %8.0f |   STALE -> stand-down" % (age*1000))
    else:         print("  %8.0f | %8.3f | quote" % (age*1000, (a-b)/2))

print("\n=== break it: thin hedge book ===")
for ex in ["Bybit","Okex"]:
    bk, m = BOOKS[ex], MIDS[ex]
    row = ["%d:%s" % (u, ("%.3f" % walk(bk,-1,u,m)) if not np.isnan(walk(bk,-1,u,m)) else "nan") for u in [1,4,8,12,20,40,80]]
    print("  " + ex.ljust(7) + " walk cost (bp): " + "  ".join(row))

=== break it: basis jump -> fill-rate ===
  k=1.0: множитель fill-rate exp(k*b) при b=1 bp = 2.72
  k=1.5: множитель fill-rate exp(k*b) при b=1 bp = 4.48
  k=3.0: множитель fill-rate exp(k*b) при b=1 bp = 20.09
  jump 2 bp при k=1.5: множитель exp(3)=20.1

=== break it: stale leader -> расширение спреда и stand-down ===
  age (ms) | half-spread (bp) | режим
         0 |    0.675 | quote
       100 |    0.875 | quote
       250 |    1.175 | quote
       500 |    1.675 | quote
      1000 |    2.675 | quote
      5010 |   STALE -> stand-down

=== break it: thin hedge book ===
  Bybit   walk cost (bp): 1:0.009  4:0.009  8:0.009  12:0.029  20:0.112  40:nan  80:nan
  Okex    walk cost (bp): 1:0.009  4:0.009  8:0.074  12:0.106  20:0.135  40:0.700  80:2.152


Stale leader ведёт себя как предписано протоколом: half-spread растёт с возрастом, а за `max_age`
функции возвращают `None` — stand-down. **Это не откат на собственный mid**: наш mid сам отстаёт на latency,
и подставить его значит заменить один stale-якорь на другой, только хуже — по данным пункта 3 разница между
якорем на Bybit и своим mid на 500 мс измеряется в разы.

Thin hedge book: у Bybit проход 40 и особенно 80 units уходит глубоко по книге (`nan` — книга тоньше
размера), стоимость хеджа растёт нелинейно, и пороговая политика, рассчитанная на дешёвый хедж, ошибается.
До $\approx 12$ units обе биржи держат стоимость на уровне спреда: тонкость книги проявляется не сразу.

**Сценарий 1 — basis jump 2 bp.** Basis jump — это ситуация, когда котировки выставлены по старому якорю,
а справедливая цена уже уехала на $b$. Прогоним это на реальных траекториях: якорь остаётся на прежнем
уровне, а истинная цена якорной биржи в середине окна уезжает на 2 bp — ровно то, что basis jump и делает с нашим
якорем. Сравниваем с включённым и выключенным детектором.

In [20]:
print("=== сценарий: basis jump 2 bp в середине окна ===")
mid_t = WIN//2
# якорь остаётся прежним (протух), истинная цена уезжает на 2 bp -> нас 'переезжает'
fair_jump = WL.copy(); fair_jump[:, mid_t:] += 2.0
anchor_stale = WL.copy(); anchor_stale[:, mid_t:] = WL[:, mid_t-1][:,None]   # якорь замер на до-скачковом уровне
scan = np.abs(np.diff(fair_jump, axis=1)).max(axis=0)
bad = np.where(scan > 1.5)[0]
print("  break detector по одному такту (порог 1.5 bp): срабатываний %d из %d — это шум, а не скачок;" % (len(bad), WIN-1))
print("     один такт не годится, нужны suspect+confirm по нескольким тактам подряд (см. протокол ниже)")
pl_ok, _, _, q_ok,_ = sim_pl(WM, WL, "lead", 1, "threshold", 2, P_SIM)
pl_nb, nf_nb, hg_nb, q_nb,_ = sim_pl(WM, fair_jump, "lead", 1, "threshold", 2, P_SIM, anchor_src=anchor_stale)
print("  baseline (якорь совпадает с ценой): pl=%.1f  |q|_sd=%.2f" % (pl_ok.mean(), q_ok.std()))
print("  basis jump 2 bp (якорь протух):     pl=%.1f  |q|_sd=%.2f  хеджировано=%.0f" % (pl_nb.mean(), q_nb.std(), hg_nb.mean()))
print("  fill-rate на стороне, ближайшей к рынку, x exp(k*b)=%.0f (k=1.5, b=2 bp) -> неконтролируемый односторонний набор" % np.exp(1.5*2.0))

print("\n=== stand-down протоколом: detect -> suspect -> confirm -> act -> re-enter ===")
# детектор по наблюдаемым признакам: лидерский фид молчит (якорь не двигался), а наш рынок ходит;
# suspect -> confirm: три таких такта подряд; act: перестаём котироваться; re-enter: только после
# обновления basis-оценки (в синтетике до конца окна не делаем — см. текст ниже)
d_anc = np.abs(np.diff(anchor_stale, axis=1)).max(axis=0)   # двигался ли якорь на такте
d_own = np.abs(np.diff(WM, axis=1)).max(axis=0)             # а наш рынок?
fired = None; run = 0
for t in range(1, WIN-1):
    run = run + 1 if (d_anc[t] < 1e-9 and d_own[t] > 0.05) else 0
    if run == 3: fired = t; break                          # confirm на третьем такте
active = np.ones(WIN); active[fired:] = 0.0                # act: stand-down до конца окна
pl_sd, nf_sd, _, q_sd,_ = sim_pl(WM, fair_jump, "lead", 1, "threshold", 2, P_SIM,
                                 anchor_src=anchor_stale, active=active)
print("  det+confirm на t=%d (скачок на t=%d)" % (fired, mid_t))
print("  pl: baseline=%.1f | jump без протокола=%.1f | с stand-down=%.1f" % (pl_ok.mean(), pl_nb.mean(), pl_sd.mean()))
print("  филлов на окно: %.0f -> %.0f; sd позиции: %.2f -> %.2f — потеряли часть прибыли, но не набрали позицию" % (
      nf_nb.mean(), nf_sd.mean(), q_nb.std(), q_sd.std()))

print("\n=== сценарий: stale leader (котировки по протухшему якорю T секунд) ===")
for stale_s in [0.5, 2.0, 5.0]:
    step = int(stale_s/0.1)
    anc = WL.copy(); anc[:, mid_t:mid_t+step] = WL[:, mid_t-1][:,None]
    pl_st,_,_,_,_ = sim_pl(WM, WL, "lead", 1, "threshold", 2, P_SIM, anchor_src=anc)
    print("  leader заморожен %4.1f с: pl=%7.1f (якорь не обновлялся)" % (stale_s, pl_st.mean()))

=== сценарий: basis jump 2 bp в середине окна ===
  break detector по одному такту (порог 1.5 bp): срабатываний 456 из 600 — это шум, а не скачок;
     один такт не годится, нужны suspect+confirm по нескольким тактам подряд (см. протокол ниже)
  baseline (якорь совпадает с ценой): pl=104.6  |q|_sd=1.34
  basis jump 2 bp (якорь протух):     pl=-420.4  |q|_sd=1.77  хеджировано=245
  fill-rate на стороне, ближайшей к рынку, x exp(k*b)=20 (k=1.5, b=2 bp) -> неконтролируемый односторонний набор

=== stand-down протоколом: detect -> suspect -> confirm -> act -> re-enter ===
  det+confirm на t=301 (скачок на t=300)
  pl: baseline=104.6 | jump без протокола=-420.4 | с stand-down=56.2
  филлов на окно: 267 -> 86; sd позиции: 1.77 -> 1.06 — потеряли часть прибыли, но не набрали позицию

=== сценарий: stale leader (котировки по протухшему якорю T секунд) ===
  leader заморожен  0.5 с: pl=  104.4 (якорь не обновлялся)
  leader заморожен  2.0 с: pl=   99.9 (якорь не обновлялся)


  leader заморожен  5.0 с: pl=   77.0 (якорь не обновлялся)


Basis jump даёт самый резкий эффект: когда якорь протух, а цена уехала на 2 bp, P&L обваливается со
$+105$ до $-420$ bp за окно — котировки оказываются не по обе стороны от справедливой цены, а целиком на одной,
и сторона, оказавшаяся ближайшей к рынку, срабатывает в $e^{kb}=20$ раз чаще, набирая одностороннюю позицию до того, как детектор
среагирует. Прогон выше показывает, что протокол работает, а не остаётся декларацией: детектор поймал молчание лидерского фида,
confirm сработал на третьем подряд подозрительном такте (в прогоне — через такт после скачка, это и есть
плата за проверку), stand-down выключил котировки — P&L окна вместо $-420$ bp даёт $+56$ bp. Цена протокола
такая: филлов за окно $267$ против $86$, то есть мы отдаём часть прибыли окна, но разброс позиции не
раздувается ($|q|$ sd $1.06$ против $1.77$ без протокола) — именно это и нужно: не набирать одностороннюю
позицию по протухшему якорю. Re-enter в прогоне сознательно отложен до конца окна: в синтетическом сценарии
Binance не подтягивается к новому уровню Bybit, и вход по «ожившему» якорю утонул бы в артефакте модели
филлов (котировки оказались бы по разные стороны от рынка фолловера, и тест мерил бы не протокол). В бою
re-enter идёт после обновления basis-оценки и подтверждения, что basis вернулся в канал. Stale leader — мягче: пока лидер заморожен, P&L падает
($105 \to 100 \to 77$ для 0.5/2/5 с), но не обваливается, потому что заморозка не создаёт двух разных цен —
она лишь лишает нас новой информации. Отсюда правила пункта 5: **threshold на приращение basis → сброс оценки;
threshold на age → stand-down; thin book требует ограничения максимального хедж-размера**, иначе стоимость
хеджа съест спред.

## Пункт 6. Report — что измерено, что допущено, чувствительность, границы

Сведём итог и разделим измеренное и допущенное.

**Что измерено (из данных).** Отличимого лидера между биржами нет: при корректной метке lead-lag
формально «впереди» Binance ($\mathrm{corr}(\Delta_t^{Binance},\Delta_{t+1}^{Bybit})=0.595$, по часам
$0.58\pm0.07$), но лаговая корреляция *выше* контемпоранной ($0.407$) — профиль, невозможный для честного
лида; на 1s лаговый сигнал исчезает ($0.10$–$0.11$ против $0.89$–$0.90$), а знак 100-мс приращений совпадает
в 90% тактов. Домен настоящего лида меньше 100 мс — за разрешением сетки. Microprice: Bybit $+0.175$ (100 мс)
$\to +0.124$ (1 с), у Binance вдвое слабее — это предсказуемость собственного mid, а не межбирженное лидерство;
тождество microprice выполняется до $5\cdot10^{-12}$ bp. Basis — сдвиг $\approx-0.18/-0.19$ bp при
$sd\approx0.83$ bp и редких скачках до $19$ bp. Hedge cost curve: малые
размеры — полспреда ($0.0085$ bp), с $12$ units растёт до $0.03$, с $20$ — до $0.11$ bp; глубина до $0.01\%$ —
$\approx25$–$27$ units на Bybit/Okex.

**Что допущено.** Интенсивности филлов $(A,k)$ не измерены (нет tape); всё P&L условно на них. Стоимость хеджа —
по снимку книги в один момент, не в реальной динамике исполнения. Модель филлов — независимые Бернулли с
постоянной интенсивностью, без кластеризации и queue priority. Data latency смоделирован как задержка якоря, а
не как сетевая задержка. Числа даны при $A=3$, $k=1.5$, $h=0.5$ bp.

**Чувствительность.** P&L растёт по $A$ и падает по $k$ (sweep); якорь и latency меняют результат
качественно (якорь $=$ fair $\gg$ свой mid; сам выбор fair иллюстративен — лидера данные не различают),
но количественно сильно зависят от параметров. Выбор micro против mid даёт
малый эффект на коротком горизонте и более заметный на длинном. Оптимум порога хеджа по идее должен зависеть от
риск-аппетита, но на наших окнах он не различимее шума ($\theta=4$ при всех трёх $\gamma$); устойчиво
главное — «threshold $\ge$ every-fill».

**Пройденные тесты.** Симметрия филлов (дрейф $0$), оптимум $1/k+h$, self-consistency стоимости хеджа
(walk против walk по той же книге), oracle-якорь, отсутствие look-ahead (prefix + perturbation), плюс семь
unit-тестов чистых функций движка — всего 12 проверок.
Stand-down протокол проверен прогоном: confirm молчания фида за три такта, P&L с $-420$ до $+56$ bp за окно.
**Счётчик испытаний.** Каждая симуляционная цифра — среднее по 300 независимым 60-секундным окнам
(стандартная ошибка среднего P&L $\approx 81/\sqrt{300}\approx 4.7$ bp — меньше эффектов якоря, но сопоставима
с различиями порогов хеджа, о чём сказано в пункте 4); sweep задержки хеджа дополнительно усреднён
по 8 сидам.

### Почему результат — верхняя граница

Три причины. Первая — **интенсивности филлов допущены, а не измерены**: при реальных $A,k$ числа могут
отличаться на порядки, а знак базовых эффектов (якорь = fair лучше своего mid) сохранится, но не их размер. Вторая —
**идеальное исполнение**: филлы исполняются ровно по котировке, без latency срабатывания, queue priority и
частичных филлов; в реальности мы получим меньше филлов и хуже цены. Третья — **хедж без footprint**: стоимость
считана по статичному снимку книги, а наш собственный проход спустил бы уровни. Плюс mark-to-market по истинной
цене — тоже оптимистичная граница, потому что истинную цену в момент котирования мы не знаем.

### Ограничения модели

Симулятор предполагает постоянные $(A,k,h)$ на всём окне, тогда как режим меняется. Мы не измерили funding, не
учли fee, не смоделировали отказ соединения и сетевой latency. Якоря и хедж-цены взяты из одного спокойного
периода, а basis может уйти в устойчивый тренд, который простой EWMA пропустит. Файл полного стакана Binance
в данных битый (перемешанные уровни с мусорными ценами, снимок crossed), поэтому hedge cost curve построена
только по Bybit и Okex; top-of-book Binance это не касается — по нему lead-lag и симуляции считаются как обычно. Наконец, вопрос «who leads» на этих данных остаётся неразрешённым: домен настоящего лида меньше 100 мс — за разрешением сетки, на 1s лаговый сигнал исчезает, а на 100 мс он неотличим от артефакта (лаговая корреляция выше контемпоранной); поэтому якорь Bybit в симуляциях — иллюстративная точка отсчёта, а не выведенное из данных лидерство. Приведённые числа — условная верхняя
граница при названных допущениях.

**Итог.** Кросс-венью движок сводится к трём ценам: затуханию сигнала, latency якоря и стоимости хеджа. Якорь, совпадающий с fair, даёт главный эффект в модели (сам выбор fair иллюстративен: отличимого лидера на данных нет), microprice — второй порядок и только при согласованном горизонте, хедж — решает, сколько спреда останется. Результат, который не выдерживает sweep, — это находка: поэтому мы отделили измеренное от
допущенного.